<div align="center">

<img src="images/Logo-Uni-Osnabrueck.jpg" width="300"/>

# Introduction to Computational Linguistics

</div>

## Regular expressions

### Cheatsheet

| Pattern | Meaning | Example match |
|---------|---------|---------------|
| `.` | Any character (except newline) | `c.t` → "cat", "cut", "c3t" |
| `*` | 0 or more of previous | `go*d` → "gd", "god", "good" |
| `+` | 1 or more of previous | `go+d` → "god", "good" (not "gd") |
| `?` | 0 or 1 of previous | `colou?r` → "color", "colour" |
| `^` | Start of string | `^Hello` → "Hello world" |
| `$` | End of string | `world$` → "Hello world" |
| `[abc]` | Any one of a, b, c | `[aeiou]` → any vowel |
| `[^abc]` | Any char NOT in set | `[^aeiou]` → any consonant |
| `\w` | Word character `[a-zA-Z0-9_]` | `\w+` → any word |
| `\d` | Digit `[0-9]` | `\d+` → "42", "2026" |
| `\s` | Whitespace | `\s+` → spaces, tabs |
| `\b` | Word boundary | `\bcat\b` → "cat" not "catch" |
| `(abc)` | Capture group | `(go)+` → "gogo" |
| `a\|b` | Either a or b | `cat\|dog` → "cat" or "dog" |
| `{n,m}` | Between n and m repetitions | `\d{2,4}` → "42", "2026" |

### Regex in Bash — `grep`

`grep` searches for lines matching a pattern.

```bash
grep 'pattern' file.txt        # basic search
grep -E 'pattern' file.txt     # extended regex (ERE)
grep -o 'pattern' file.txt     # print only the matched part
grep -i 'pattern' file.txt     # case-insensitive
```

In [17]:
%%bash
TEXT="The cat sat on the mat. A cat in a hat. 2 cats, 3 dogs."

# Lines containing "cat"
echo "$TEXT" | grep -o 'cat'

# Words starting with capital letter
echo "$TEXT" | grep -oE '\b[A-Z][a-z]+'

# All numbers
echo "$TEXT" | grep -oE '\d+' || echo "$TEXT" | grep -oE '[0-9]+'

# Words ending in "at"
echo "$TEXT" | grep -oE '\b\w+at\b'

cat
cat
cat
The
2
3
cat
sat
mat
cat
hat


### Regex in Python — `re` module

The four functions you'll use most:

| Function | What it does |
|----------|-------------|
| `re.search(pattern, text)` | Find first match anywhere in text |
| `re.match(pattern, text)` | Match only at the **start** of text |
| `re.findall(pattern, text)` | Return **all** matches as a list |
| `re.sub(pattern, replacement, text)` | **Replace** matches |

In [18]:
import re

text = "The cat sat on the mat. A cat in a hat. 2 cats, 3 dogs."

# --- findall: extract all matches ---
words_ending_at = re.findall(r'\b\w+at\b', text)
print("Words ending in 'at':", words_ending_at)

numbers = re.findall(r'\d+', text)
print("Numbers found:", numbers)

# --- search: find first match ---
match = re.search(r'\bcat\b', text)
if match:
    print(f"First 'cat' at position {match.start()}–{match.end()}")

# --- sub: replace matches ---
censored = re.sub(r'\bcat\b', '***', text)
print("Censored:", censored)

# --- groups: extract parts of a match ---
date_text = "Published on 2026-05-12 and updated on 2026-05-15"
dates = re.findall(r'(\d{4})-(\d{2})-(\d{2})', date_text)
print("Dates (year, month, day):", dates)

Words ending in 'at': ['cat', 'sat', 'mat', 'cat', 'hat']
Numbers found: ['2', '3']
First 'cat' at position 4–7
Censored: The *** sat on the mat. A *** in a hat. 2 cats, 3 dogs.
Dates (year, month, day): [('2026', '05', '12'), ('2026', '05', '15')]


### Practice

Try modifying the examples above, or work on these:

In [19]:
sentences = [
    "She sells sea-shells by the sea shore.",
    "Call me at +1-800-555-0199 or email me at hello@example.com",
    "The price is $4.99, but with discount it's $3.50",
]

# 1. Extract all hyphenated words from sentences[0]
# Expected: ['sea-shells', 'sea-shore'] (approximately)
pattern_hyphen = r'...'  # <- your pattern here
print(re.findall(pattern_hyphen, sentences[0]))

# 2. Extract the phone number from sentences[1]
pattern_phone = r'...'  # <- your pattern here
print(re.findall(pattern_phone, sentences[1]))

# 3. Extract all prices (with $) from sentences[2]
pattern_price = r'...'  # <- your pattern here
print(re.findall(pattern_price, sentences[2]))

['She', ' se', 'lls', ' se', 'a-s', 'hel', 'ls ', 'by ', 'the', ' se', 'a s', 'hor']
['Cal', 'l m', 'e a', 't +', '1-8', '00-', '555', '-01', '99 ', 'or ', 'ema', 'il ', 'me ', 'at ', 'hel', 'lo@', 'exa', 'mpl', 'e.c']
['The', ' pr', 'ice', ' is', ' $4', '.99', ', b', 'ut ', 'wit', 'h d', 'isc', 'oun', 't i', "t's", ' $3', '.50']


## Word tokenization

### Space-based tokenization

Splitting on whitespace is the simplest approach — but it's naive. Punctuation sticks to words, contractions break unpredictably, and you get noisy tokens like `"father's"`, `"weep!"`, `"God,"`.

In [20]:
import re
from collections import Counter

with open('sources/shakespeare.txt', 'r') as f:
    raw = f.read()

# --- Naive: split on whitespace ---
naive_tokens = raw.split()
print(f"Naive split → {len(naive_tokens):,} tokens")
print("Sample:", naive_tokens[300:308])

# Problem: punctuation is attached
dirty = [t for t in naive_tokens if not t.isalpha()]
print(f"\nTokens with noise (punctuation, numbers): {len(dirty):,}")
print("Examples:", dirty[10:18])

# --- Better: regex word tokenizer ---
word_tokens = re.findall(r"[a-zA-Z]+(?:'[a-z]+)?", raw)
print(f"\nRegex tokenizer → {len(word_tokens):,} tokens")
print("Sample:", word_tokens[300:308])

# Vocabulary size
vocab = set(t.lower() for t in word_tokens)
print(f"\nVocabulary (unique words): {len(vocab):,}")

Naive split → 1,068,193 tokens
Sample: ['lack', 'of', 'work.', 'Would,', 'for', 'the', "King's", 'sake,']

Tokens with noise (punctuation, numbers): 281,418
Examples: ['FLORENCE.', 'DIANA,', 'VIOLENTA,', 'MARIANA,', 'Lords,', 'Officers,', 'Soldiers,', 'etc.,']

Regex tokenizer → 1,073,645 tokens
Sample: ['lack', 'of', 'work', 'Would', 'for', 'the', "King's", 'sake']

Vocabulary (unique words): 26,122


### Simple Tokenization in UNIX

#### The first step: tokenizing

`tr -sc 'A-Za-z' '\n'` — keep only letters, replace everything else with a newline. One word per line.

In [21]:
%%bash
tr -sc 'A-Za-z' '\n' < sources/shakespeare.txt | head -15


ALLS
WELL
THAT
ENDS
WELL
by
William
Shakespeare
Dramatis
Personae
KING
OF
FRANCE
THE


#### The second step: sorting

Pipe into `sort` — brings identical words together so we can count them.

In [22]:
%%bash
tr -sc 'A-Za-z' '\n' < sources/shakespeare.txt | sort | head -15


A
A
A
A
A
A
A
A
A
A
A
A
A
A


#### More counting

Add `uniq -c` to count consecutive duplicates, then `sort -rn` to rank by frequency.

In [23]:
%%bash
tr -sc 'A-Za-z' '\n' < sources/shakespeare.txt \
  | tr 'A-Z' 'a-z' \
  | sort \
  | uniq -c \
  | sort -rn \
  | head -20

33233 the
31284 and
27298 i
24291 to
20871 of
17579 a
16925 you
15216 my
13659 that
13192 in
11267 is
10405 d
10360 not
9423 with
9409 for
9387 it
9382 me
9378 s
8465 be
8304 his


### Tokenization in languages without spaces

In Chinese, Japanese, and Thai, words are **not separated by spaces**. Splitting on whitespace gives you nothing useful — you get full sentences as single "tokens". Segmentation requires dedicated algorithms or dictionaries.

| Language | Sentence | Space-split tokens |
|----------|----------|--------------------|
| English | `the cat sat` | `["the", "cat", "sat"]` ✓ |
| Chinese | `猫坐在垫子上` | `["猫坐在垫子上"]` ✗ |

In [24]:
en = "The cat sat on the mat"
zh = "猫坐在垫子上"          # "The cat sat on the mat" in Chinese

print("English — split():", en.split())
print(f"  → {len(en.split())} tokens\n")

print("Chinese — split():", zh.split())
print(f"  → {len(zh.split())} tokens  (whole sentence = one 'token'!)")
print()

# Chinese characters are already word-like units — iterate over chars as a proxy
print("Chinese — character-level (rough baseline):", list(zh))
print(f"  → {len(zh)} characters")

English — split(): ['The', 'cat', 'sat', 'on', 'the', 'mat']
  → 6 tokens

Chinese — split(): ['猫坐在垫子上']
  → 1 tokens  (whole sentence = one 'token'!)

Chinese — character-level (rough baseline): ['猫', '坐', '在', '垫', '子', '上']
  → 6 characters


### Practice

In [ ]:
import re
from collections import Counter

with open('sources/shakespeare.txt', 'r') as f:
    raw = f.read()

# 1. How many unique words does Shakespeare use?
#    Use the regex tokenizer from above: r"[a-zA-Z]+(?:'[a-z]+)?"
#    Then count unique lowercased forms.
tokens = ...  # your code here
vocab  = ...  # your code here
print("Unique words:", len(vocab))

# 2. What are the 10 most common words?
#    Hint: Counter(tokens).most_common(10)
most_common = ...  # your code here
print("Top 10:", most_common)

# 3. How many words appear only ONCE (hapax legomena)?
#    Hapax = words with frequency 1 — a classic measure of vocabulary richness.
hapax = ...  # your code here
print("Hapax legomena:", len(hapax))

# 4. Type-Token Ratio (TTR): unique words / total tokens
#    A rough measure of vocabulary diversity (higher = more varied).
ttr = ...  # your code here
print(f"TTR: {ttr:.4f}")